<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 2 &middot; Week 5 &middot; Part 2 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Softmax and Multiclass Classification

A sigmoid output answers one yes-or-no question. Many problems have more than two answers: which
digit, which leaf disease, which kind of fabric defect. The **softmax** output layer turns $N$
scores into $N$ probabilities that sum to 1, and its loss, cross-entropy, generalises the
logistic loss.

This part implements softmax and its loss in NumPy, checks them against the lecture and against
TensorFlow, and then breaks them: you will watch the obvious implementation return `nan`, and see
why TensorFlow wants the layer to output **logits**. It ends by training a four-class network
and drawing its decision regions.

**Companion theory lecture:** CSE 815, Week 5, Part 2 (Multiclass Classification and
Back-propagation).

## What you will be able to do

1. Implement softmax, looped and vectorised, and reproduce the lecture's worked example.
2. Show that softmax with two classes is logistic regression.
3. Compute the cross-entropy loss and match it to `SparseCategoricalCrossentropy`.
4. Explain, with a failing example, why softmax should be computed with the maximum subtracted,
   and why `from_logits=True` gives better losses than a softmax output layer.
5. Train a multiclass network in TensorFlow, turn its logits into probabilities, and plot its
   decision regions.

---

## 1. Setup

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow's start-up chatter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)
keras.utils.set_random_seed(816)      # seeds Python, NumPy and TensorFlow together

BLUE, TEAL, AMBER, ROSE, GRAY = "#123B63", "#1B7F79", "#C97B17", "#9B2C4B", "#4A5560"

print("NumPy", np.__version__, "  TensorFlow", tf.__version__)

---

## 2. The softmax function

For scores $\mathbf{z} = [z_1, \dots, z_N]$,

$$a_j = \frac{e^{z_j}}{\sum_{k=1}^{N} e^{z_k}}, \qquad j = 1, \dots, N.$$

Start with a loop, so that the formula is visible line by line.

In [ ]:
def my_softmax_loop(z):
    ez = np.zeros(len(z))
    for j in range(len(z)):
        ez[j] = np.exp(z[j])
    total = 0.0
    for j in range(len(z)):
        total += ez[j]
    a = np.zeros(len(z))
    for j in range(len(z)):
        a[j] = ez[j] / total
    return a


z = np.array([2.0, 1.0, 0.1])
a = my_softmax_loop(z)
print("e^z    :", np.exp(z), "  sum", round(np.exp(z).sum(), 4))
print("softmax:", a, "  sum", a.sum())

assert np.allclose(np.exp(z), [7.3891, 2.7183, 1.1052], atol=5e-5)
assert np.allclose(a, [0.6590, 0.2424, 0.0986], atol=5e-5)
assert np.isclose(a.sum(), 1.0)
print("\nThe lecture's table: 0.6590, 0.2424, 0.0986.")

The vectorised version works on a whole batch at once: `Z` has one row per example and one
column per class, and each **row** is normalised separately. `axis=1, keepdims=True` keeps the
row sums as a column, so that broadcasting divides each row by its own sum.

In [ ]:
def my_softmax(Z):
    """Z: (m, N) scores  ->  (m, N) probabilities; each row sums to 1."""
    EZ = np.exp(Z)
    return EZ / EZ.sum(axis=1, keepdims=True)


Z_batch = np.array([[2.0, 1.0, 0.1],
                    [0.0, 0.0, 0.0],
                    [-1.0, 5.0, 2.0]])
A_batch = my_softmax(Z_batch)
print(A_batch)
print("row sums:", A_batch.sum(axis=1))

assert np.allclose(A_batch[0], my_softmax_loop(Z_batch[0]))
assert np.allclose(A_batch[1], 1 / 3)                 # equal scores, equal probabilities
assert np.allclose(A_batch, tf.nn.softmax(Z_batch).numpy())
print("\nLoop, vectorised and tf.nn.softmax agree.")

### Three properties worth checking

1. **Adding the same constant to every score changes nothing**, because $e^{z_j + c} =
   e^c e^{z_j}$ and $e^c$ cancels. (Section 4 relies on this.)
2. **Softmax couples the outputs.** Raise one score and *every other* probability falls. No
   other activation you have met behaves like this.
3. **With $N = 2$ it is logistic regression:** $a_1 = g(z_1 - z_2)$, where $g$ is the sigmoid.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


# 1. Shift invariance
z = np.array([[2.0, 1.0, 0.1]])
for c in [-5.0, 10.0, 100.0]:
    assert np.allclose(my_softmax(z + c), my_softmax(z))
print("1. softmax(z + c) = softmax(z) for c = -5, 10, 100")

# 2. Coupling: raise z_1 only
for z1 in [2.0, 3.0, 4.0]:
    print(f"2. z = [{z1}, 1, 0.1]  ->  a =", my_softmax(np.array([[z1, 1.0, 0.1]]))[0])

# 3. Two classes
z2 = np.array([[2.0, 0.5]])
print(f"3. softmax([2, 0.5])[0] = {my_softmax(z2)[0, 0]:.4f},  sigmoid(1.5) = {sigmoid(1.5):.4f}")
assert np.isclose(my_softmax(z2)[0, 0], sigmoid(1.5))
assert np.isclose(sigmoid(1.5), 0.8176, atol=5e-5)

---

## 3. The loss: cross-entropy

Only the probability assigned to the **correct** class counts:

$$L(\mathbf{a}, y) = -\log a_y.$$

In TensorFlow this is `SparseCategoricalCrossentropy`. *Sparse* means the label is stored as one
integer, $0$ to $N-1$ in code, rather than as a one-hot vector. Note the shift: the lecture's
classes $1, 2, 3$ are `0, 1, 2` in Python.

In [ ]:
a = my_softmax(np.array([[2.0, 1.0, 0.1]]))[0]
losses = -np.log(a)
for j in range(3):
    print(f"true class {j}:  L = -log a_{j} = {losses[j]:.4f}")
assert np.allclose(losses, [0.4170, 1.4170, 2.3170], atol=5e-5)

# The gaps between losses are the gaps between scores: -log a_j = log(sum e^z) - z_j.
lse = np.log(np.exp([2.0, 1.0, 0.1]).sum())
assert np.allclose(losses, lse - np.array([2.0, 1.0, 0.1]))

scce = keras.losses.SparseCategoricalCrossentropy()
for j in range(3):
    assert np.isclose(float(scce(np.array([j]), a.reshape(1, -1))), losses[j], atol=1e-5)
print("\nSparseCategoricalCrossentropy returns the same three numbers.")

---

## 4. When the obvious code fails

### Round-off

Floating-point numbers carry a fixed number of significant digits. A small number computed as the
difference of two numbers near 1 loses most of them:

In [ ]:
x1 = 2.0 / 10000
x2 = 1 + 1 / 10000 - (1 - 1 / 10000)
print(f"x1 = {x1:.20f}")
print(f"x2 = {x2:.20f}")
print(f"relative error of x2: {abs(x2 - x1) / x1:.1e}")

x2_32 = np.float32(1) + np.float32(1 / 10000) - (np.float32(1) - np.float32(1 / 10000))
print(f"in float32, as TensorFlow computes: {x2_32:.20f}  "
      f"(relative error {abs(x2_32 - x1) / x1:.1e})")

In float64 the error is in the 14th digit; in float32, TensorFlow's default, it is already in
the 4th or 5th. Softmax has a worse problem than round-off.

### Overflow

Scores of a few hundred are not unusual in an untrained or badly scaled network. $e^{1000}$ is
far beyond the largest float, about $1.8 \times 10^{308}$.

In [ ]:
z_big = np.array([[1000.0, 999.0, 998.0]])
with np.errstate(over="ignore", invalid="ignore"):
    print("np.exp(z)          :", np.exp(z_big))
    print("naive my_softmax(z):", my_softmax(z_big))

`inf / inf` is `nan`, and one `nan` in a cost poisons every gradient after it. The fix is the
shift invariance of section 2: subtract $c = \max_k z_k$ first. Every exponent is then
$\le 0$, so nothing overflows, and the largest term is exactly $e^0 = 1$, so the sum cannot
underflow to 0 either.

For the loss, do better still: never form $a_y$ and then take its log. Simplify first,

$$-\log a_y = \log\sum_k e^{z_k} - z_y = \Bigl(c + \log\sum_k e^{z_k - c}\Bigr) - z_y.$$

In [ ]:
def stable_softmax(Z):
    Zs = Z - Z.max(axis=1, keepdims=True)
    EZ = np.exp(Zs)
    return EZ / EZ.sum(axis=1, keepdims=True)


def log_softmax(Z):
    """log of softmax(Z), computed without forming softmax(Z)."""
    c = Z.max(axis=1, keepdims=True)
    return Z - (c + np.log(np.exp(Z - c).sum(axis=1, keepdims=True)))


print("stable softmax of [1000, 999, 998]:", stable_softmax(z_big)[0])
assert np.allclose(stable_softmax(z_big), stable_softmax(z_big - 998))   # = softmax([2, 1, 0])

L_big = -log_softmax(z_big)[0, 0]
print(f"loss if the true class is 0: {L_big:.4f}")
assert np.isclose(L_big, np.log(1 + np.exp(-1) + np.exp(-2)))
assert np.isclose(L_big, 0.4076, atol=5e-5)
print("Finite, and equal to log(1 + e^-1 + e^-2) = 0.4076, as on the lecture slide.")

### What this means for TensorFlow: `from_logits=True`

There are two ways to build the output layer of a multiclass network.

| | output layer | loss |
|---|---|---|
| **A** (fragile) | `Dense(N, activation="softmax")` | `SparseCategoricalCrossentropy()` |
| **B** (recommended) | `Dense(N, activation="linear")` | `SparseCategoricalCrossentropy(from_logits=True)` |

In **A** the layer outputs probabilities and the loss takes their log. A probability that
rounds to 0 has an infinite log, so Keras quietly **clips** probabilities to at least
$10^{-7}$ before taking it. In **B** the layer outputs the scores $\mathbf{z}$ &mdash; the
**logits** &mdash; and the loss evaluates $\log\sum_k e^{z_k} - z_y$ directly, as above.

Compare them on an example that is confidently **wrong**: logits $[0, 50]$, true class 0.

In [ ]:
logits = np.array([[0.0, 50.0]], dtype="float32")
y_true = np.array([0])

loss_A = float(keras.losses.SparseCategoricalCrossentropy()(y_true, tf.nn.softmax(logits)))
loss_B = float(keras.losses.SparseCategoricalCrossentropy(from_logits=True)(y_true, logits))
exact = float(-log_softmax(logits.astype(float))[0, 0])

print(f"exact loss            : {exact:.4f}")
print(f"A: softmax, then loss : {loss_A:.4f}")
print(f"B: from_logits=True   : {loss_B:.4f}")
print(f"-log(1e-7)            : {-np.log(1e-7):.4f}")

assert np.isclose(loss_B, exact, atol=1e-3)
assert np.isclose(loss_A, -np.log(1e-7), atol=1e-3)

Version **A** reports a loss of 16.12 however wrong the model is: the clip has capped it, and the
gradient beyond the cap is zero, so a badly wrong example stops pulling on the weights. Version
**B** reports the true 50. Use **B**. The cost is one extra line when you want probabilities:
the model now outputs logits, and `tf.nn.softmax` converts them.

---

## 5. A four-class network

Four clusters of points in the plane, one per class, with some overlap between neighbours.

In [ ]:
from sklearn.datasets import make_blobs

centres = np.array([[-2.5, 2.5], [2.5, 2.5], [-2.5, -2.5], [2.5, -2.5]])
X, y = make_blobs(n_samples=400, centers=centres, cluster_std=1.3, random_state=816)
print("X", X.shape, "  y", y.shape, "  classes", np.unique(y), "  counts", np.bincount(y))

rng = np.random.default_rng(816)
idx = rng.permutation(len(y))
train, test = idx[:300], idx[300:]
X_train, y_train, X_test, y_test = X[train], y[train], X[test], y[test]

COLOURS = [BLUE, ROSE, TEAL, AMBER]
MARKERS = ["o", "x", "^", "s"]
fig, ax = plt.subplots(figsize=(5, 4.3))
for k in range(4):
    ax.scatter(*X_train[y_train == k].T, c=COLOURS[k], marker=MARKERS[k], s=18, label=f"class {k}")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.legend(fontsize=8)
ax.set_title("Training set: 300 points, 4 classes", fontsize=10)
plt.show()

The recommended architecture: ReLU hidden layers, a **linear** output layer with one unit per
class, and `from_logits=True`.

In [ ]:
keras.utils.set_random_seed(816)
model = Sequential([
    Input(shape=(2,)),
    Dense(units=25, activation="relu", name="L1"),
    Dense(units=15, activation="relu", name="L2"),
    Dense(units=4, activation="linear", name="L3"),     # logits
], name="four_blobs")
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
)
model.summary()
history = model.fit(X_train, y_train, epochs=100, verbose=0)

loss = history.history["loss"]
for epoch in [1, 10, 50, 100]:
    print(f"epoch {epoch:3d}   cost {loss[epoch - 1]:.4f}")
assert model.count_params() == (2 * 25 + 25) + (25 * 15 + 15) + (15 * 4 + 4) == 529

### From logits to predictions

`model.predict` returns logits: any real numbers, not summing to 1. For the predicted class you
do not need softmax at all, because softmax preserves order: $\arg\max_j z_j = \arg\max_j a_j$.
For probabilities, apply it.

In [ ]:
logits_test = model.predict(X_test, verbose=0)               # (100, 4)
probs_test = tf.nn.softmax(logits_test).numpy()

print("first three test points")
print("logits:\n", logits_test[:3])
print("probabilities:\n", probs_test[:3])
print("labels:", y_test[:3])

assert np.allclose(probs_test.sum(axis=1), 1, atol=1e-5)
assert np.array_equal(logits_test.argmax(axis=1), probs_test.argmax(axis=1))
assert np.allclose(probs_test, stable_softmax(logits_test.astype(float)), atol=1e-6)

acc_train = float((model.predict(X_train, verbose=0).argmax(axis=1) == y_train).mean())
acc_test = float((logits_test.argmax(axis=1) == y_test).mean())
print(f"\ntraining accuracy {acc_train:.3f}   test accuracy {acc_test:.3f}")
assert acc_test >= 0.85

### Decision regions

Colour every point of the plane by the class the network predicts there. Each boundary is
assembled from straight pieces &mdash; one per ReLU hinge in play.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-7, 7, 300), np.linspace(-7, 7, 300))
grid = np.c_[g1.ravel(), g2.ravel()]
grid_logits = model.predict(grid, verbose=0)
region = grid_logits.argmax(axis=1).reshape(g1.shape)
confidence = tf.nn.softmax(grid_logits).numpy().max(axis=1).reshape(g1.shape)

from matplotlib.colors import ListedColormap
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
axes[0].contourf(g1, g2, region, levels=[-0.5, 0.5, 1.5, 2.5, 3.5],
                 cmap=ListedColormap(COLOURS), alpha=0.25)
for k in range(4):
    axes[0].scatter(*X_test[y_test == k].T, c=COLOURS[k], marker=MARKERS[k], s=18)
wrong = logits_test.argmax(axis=1) != y_test
axes[0].scatter(*X_test[wrong].T, facecolors="none", edgecolors="k", s=110, lw=1.2,
                label="misclassified")
axes[0].legend(fontsize=8); axes[0].set_title("Predicted class, with the test set", fontsize=10)
cs = axes[1].contourf(g1, g2, confidence, levels=np.linspace(0.25, 1, 16), cmap="Greys")
fig.colorbar(cs, ax=axes[1], label="largest softmax probability")
axes[1].set_title("How sure the network is", fontsize=10)
for ax in axes:
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
plt.tight_layout()
plt.show()
print(f"misclassified test points: {int(wrong.sum())} of {len(y_test)}")

In the reference run the network scores `0.97` on the training set and `0.95` on the test
set; the five test points it misses lie in the overlap between neighbouring clusters. It is
least sure along the boundaries, and nowhere is its largest probability below
$1/4$ (the value when all four classes tie). Far from the data it remains confident: a softmax
classifier has no way of saying "this looks like none of the classes".

### Checkpoint 1

Compute by hand &mdash; with `my_softmax_loop`, not TensorFlow &mdash; the softmax of
$\mathbf{z} = [1,\ 0,\ -1]$ and the loss for each possible true class (lecture, descriptive
question 2). Check your numbers with `tf.nn.softmax` and `SparseCategoricalCrossentropy`.

*Expected:* $\mathbf{a} \approx$ `[0.6652, 0.2447, 0.0900]`; losses `0.4076`, `1.4076`,
`2.4076`. Why is the first loss the same as for $[1000, 999, 998]$?

In [ ]:
# Your code here

### Checkpoint 2

Write `softmax_xent(Z, y)` that returns the **mean** cross-entropy for a batch of logits `Z`,
shape `(m, N)`, and integer labels `y`, shape `(m,)`, using `log_softmax` and **no loop**
(hint: `log_softmax(Z)[np.arange(m), y]` picks the right entry from each row). Then:

1. Test it against `SparseCategoricalCrossentropy(from_logits=True)` on
   `Z = np.random.default_rng(1).normal(size=(500, 6)) * 100` with
   `y = np.random.default_rng(2).integers(0, 6, 500)`. Logits this large would overflow the
   naive formula.
2. Use it on `model`'s training-set logits and compare with
   `model.evaluate(X_train, y_train, verbose=0)`.

*Expected:* agreement to about `1e-4` relative in both tests (TensorFlow works in float32).
The training-set cost in the reference run is about `0.090`.

In [ ]:
# Your code here

---

## 6. Recap

- Softmax: $a_j = e^{z_j} / \sum_k e^{z_k}$. Outputs are positive and sum to 1; raising one
  score lowers every other probability. For $[2, 1, 0.1]$ it gives `0.6590, 0.2424, 0.0986`.
- With two classes, softmax is the sigmoid of the difference of the scores.
- Cross-entropy, $-\log a_y$, penalises only the true class's probability. TensorFlow's
  `SparseCategoricalCrossentropy` takes integer labels.
- Exponentials overflow: subtract the largest score first; compute $\log$-softmax directly.
- Output **logits** (linear last layer) and use `from_logits=True`; a softmax layer plus the
  default loss clips probabilities at $10^{-7}$ and caps the loss at `16.12`. Apply
  `tf.nn.softmax` to the logits only when you need probabilities.
- A $2 \to 25 \to 15 \to 4$ ReLU network separates the four blobs with piecewise-linear
  boundaries.

### Exercises to hand in

1. **The fragile version.** Retrain the four-class network as version A (softmax output layer,
   default loss), same seed. Compare accuracy and the learning curve with version B. Then
   multiply the inputs by 50 and train both again: what happens, and why?
2. **One-hot labels.** Convert `y_train` to one-hot vectors with `keras.utils.to_categorical`
   and train with `CategoricalCrossentropy(from_logits=True)`. Show that the training cost is
   identical to the sparse version at every epoch. When would you prefer one-hot labels?
3. **Multi-label.** Make a data set where each point can belong to *several* classes: label
   $[x_1 > 0,\ x_2 > 0,\ x_1 + x_2 > 3]$ for $x$ uniform on $[-5, 5]^2$. Train a network with
   three sigmoid outputs and `BinaryCrossentropy(from_logits=True)`. Then train a 3-class
   softmax network on the same points using only the first true label of each, and explain from
   the outputs why softmax is the wrong model here.
4. **Temperature.** Compute `stable_softmax(z / T)` for $\mathbf{z} = [2, 1, 0.1]$ and
   $T = 0.1, 0.5, 1, 2, 10$. Describe the limit as $T \to 0$ and as $T \to \infty$, and say
   why this is called a "soft" max.

### Next

**Part 3 &mdash; Derivatives and back-propagation:** how `fit` finds every gradient, written out
by hand and checked against TensorFlow; and the Adam optimiser, built from scratch.